In [1]:
import pandas as pd
import numpy as np

# ===================== [Core Settings] Modify parameters here =====================
delta_t_target = 10    # Target time lag step (e.g., 10)
multiscale = True     # True=Multi-scale concatenation (1~10) | False=Single scale (only target step)
input_file = "20260323_Matched.csv"  # Supported formats: csv / xlsx / xls
# ===================================================================================

# Difference calculation function (unchanged)
def compute_difference(data, delta_t=1, relative=False, eps=1e-6):
    if delta_t < 1 or delta_t >= len(data):
        raise ValueError("delta_t must be >= 1 and less than the total number of samples")
    x_future = data[delta_t:]
    x_now = data[:-delta_t]
    if relative:
        diff = (x_future - x_now) / (x_now + eps)
    else:
        diff = x_future - x_now
    return diff

from tqdm.auto import tqdm

# ===================== Adaptive File Loader =====================
print(f"Loading input file: {input_file}")
if input_file.endswith(".csv"):
    df = pd.read_csv(input_file)
elif input_file.endswith((".xlsx", ".xls")):
    df = pd.read_excel(input_file)
else:
    raise ValueError("Unsupported file format! Only .csv / .xlsx / .xls are accepted")

# Data cleaning: retain numeric columns + fill missing values
df = df.select_dtypes(include=[np.number])
df = df.ffill().replace([np.inf, -np.inf], 0)
data = df.values

# ============== Main Calculation Logic ==============
if multiscale:
    # Multi-scale: compute all lags from 1 to delta_t_target and concatenate results
    all_diffs = []
    for dt in tqdm(range(1, delta_t_target + 1)):
        diff = compute_difference(data, delta_t=dt, relative=False)
        diff_df = pd.DataFrame(diff, columns=df.columns)
        all_diffs.append(diff_df)
    final_df = pd.concat(all_diffs, axis=0, ignore_index=True)
    filename = f"20260323_matched_derivative_multiscale_{delta_t_target}.csv"
else:
    # Single scale: only compute the target delta_t lag
    diff = compute_difference(data, delta_t=delta_t_target, relative=False)
    final_df = pd.DataFrame(diff, columns=df.columns)
    filename = f"20260323_matched_derivative_single_scale_{delta_t_target}_simplified.csv"

# Export output file
final_df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"File saved successfully: {filename}")

正在读取文件: 20260323_Matched.csv


  0%|          | 0/10 [00:00<?, ?it/s]

✅ 保存成功: 20260323_matched_derivative_multiscale_10.csv
